# Tutorial do pyHidroWeb

**pyHidroWeb** é uma biblioteca Python para obter, padronizar, analisar, visualizar e
exportar dados hidrológicos da **ANA — Agência Nacional de Águas e Saneamento Básico**.

A partir da versão **2.0**, o pyHidroWeb é uma tradução para Python do pacote R
[hydroDataBR](https://github.com/hydrostat/hydroDataBR) (hydro_stat/UFMG, MIT), cobrindo
toda a sua API pública.

Este tutorial está organizado em duas trilhas:

- **Offline** (células executáveis): usam o banco interno embutido (retrato ANA de
  junho/2026, com 37.584 estações) e arquivos de exemplo — funcionam sem internet nem
  credenciais.
- **Online** (blocos de código não executados): aquisição ao vivo pela API da ANA;
  exigem rede e, em alguns casos, credenciais próprias cadastradas na ANA.

## 1. Instalação

```bash
pip install pyhydroweb              # núcleo: pandas, numpy, requests, pyarrow
pip install pyhydroweb[plot]        # + matplotlib (gráficos)
pip install pyhydroweb[geo]         # + geopandas/shapely (shapefiles)
pip install pyhydroweb[xarray]      # + xarray (saída legada)
pip install pyhydroweb[vertices]    # + pyreadr (vértices de seções transversais)
pip install pyhydroweb[all]
```

## 2. O contrato diário padronizado

Toda fonte de dados diários — XML legado, JSON da API, CSV/ZIP do HidroWeb — é
normalizada para o **mesmo DataFrame de 8 colunas**:

| coluna | descrição |
|---|---|
| `station_code` | código da estação (str) |
| `date` | data da observação |
| `variable` | `discharge` (m³/s), `stage` (cm) ou `rainfall` (mm) |
| `value` | valor medido |
| `unit` | unidade da variável |
| `consistency_level` | `"2"` = consistido (preferido), `"1"` = bruto |
| `source_status` | flag de qualidade da fonte |
| `source` | proveniência do dado |

Com `merge_consistency=True` (padrão), mantém-se uma linha por
(estação, dia, variável), preferindo o dado consistido.

## 3. Encontrando estações (offline, banco interno)

`filter_ana_stations()` filtra o inventário embutido (37.584 estações reais). Combine
critérios: UF, município, bacia, tipo, produto disponível, situação de operação e um
padrão (regex) no nome.

In [ ]:
import pyhydroweb as ph

# Estações fluviométricas em operação no Piauí, com dados de vazão
stations = ph.filter_ana_stations(
    state_code="PI",
    product="discharge",
    is_operating=True,
)
print(f"{len(stations)} estações encontradas")
stations[["station_code", "station_name", "latitude", "longitude", "drainage_area_km2"]].head()

In [ ]:
# Busca por nome (regex, case-insensitive)
ph.filter_ana_stations(name_pattern="luzil")[
    ["station_code", "station_name", "state_code"]
]

### 3.1 Produtos disponíveis por estação

`table_hydro_data(..., "station_products")` lista, para uma estação, quais produtos o
inventário indica como disponíveis e seus períodos.

In [ ]:
ph.table_hydro_data(None, "station_products", station_code="34879500")

## 4. Inventário ao vivo (online — requer credenciais)

A rota autenticada `/HidroInventarioEstacoes/v1` traz o inventário atualizado. Requer
credenciais próprias cadastradas na ANA, fornecidas por variáveis de ambiente:

```bash
export ANA_HIDROWEBSERVICE_IDENTIFIER="seu-identificador"
export ANA_HIDROWEBSERVICE_PASSWORD="sua-senha"
```

```python
import pyhydroweb as ph

token = ph.ana_authenticate()            # lê as variáveis de ambiente
inventory = ph.get_ana_stations(state_code="MG", token=token)
inventory.head()
```

O objeto `token` mantém o valor apenas em memória e o oculta no `repr`.

## 5. Baixando séries diárias (online — WebService legado, sem credenciais)

`get_ana_data` com a fonte padrão `webservice` usa a rota pública HTTPS
`HidroSerieHistorica` (não requer credenciais). Produtos: `daily_discharge`,
`daily_stage`, `daily_rainfall`.

```python
daily = ph.get_ana_data(
    "daily_discharge",
    station_code="34879500",         # LUZILÂNDIA (PI)
    start_date="2015-01-01",
    end_date="2020-12-31",
)
daily.head()
```

Várias estações de uma vez, com relatório por estação:

```python
batch = ph.get_ana_data_batch(
    "daily_discharge",
    ["34879500", "34880000"],
    start_date="2020-01-01",
    end_date="2020-12-31",
)
print(batch["request_report"])
daily_34879500 = batch["results"]["34879500"]
```

## 6. Lendo arquivos locais do HidroWeb (offline)

Se você já baixou arquivos do portal, leia-os diretamente. Todos retornam o contrato
diário padronizado. Abaixo usamos os arquivos de exemplo que acompanham o repositório;
se não estiverem disponíveis, criamos uma série sintética para o restante do tutorial
continuar funcionando.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Os exemplos ficam em ../tests/fixtures relativamente a esta pasta (tutorials/)
fixtures = Path.cwd().parent / "tests" / "fixtures"
discharge_csv = fixtures / "hidroweb_discharge.csv"

if discharge_csv.exists():
    sample = ph.read_hidroweb(discharge_csv)
    print("Lido do arquivo de exemplo:", discharge_csv.name)
else:
    print("Arquivo de exemplo não encontrado; seguindo com dados de exemplo embutidos.")
    sample = None

if sample is not None:
    display(sample.head())

`read_ana_xml`, `read_ana_json` e `read_hidroweb_cross_sections` cobrem os demais
formatos (XML legado salvo em disco, JSON da API salvo em disco e o
`PerfilTransversal.csv` de seções transversais):

```python
daily_xml = ph.read_ana_xml("HidroSerieHistorica_56460000.xml")
daily_json = ph.read_ana_json("serie_56460000.json")
xs = ph.read_hidroweb_cross_sections("estacao_56460000.zip")   # -> {"sections", "vertices"}
```

## 7. Análises hidrológicas (offline)

`analyze_hydro_data(data, analysis=...)` opera sobre qualquer série no contrato diário.
Para o tutorial rodar de forma determinística e offline, geramos abaixo uma série
sintética de 6 anos (vazão, cota e chuva) — na prática, você passaria o resultado de
`get_ana_data`/`read_hidroweb`.

In [ ]:
# Série sintética determinística (2015-2020) só para demonstrar as análises
rng = np.random.default_rng(42)
dates = pd.date_range("2015-01-01", "2020-12-31", freq="D")
season = np.sin(2 * np.pi * dates.dayofyear / 365.25)

def frame(variable, unit, base, amp, noise):
    values = np.clip(base + amp * season + rng.normal(0, noise, len(dates)), 0, None)
    return pd.DataFrame(
        {
            "station_code": "34879500",
            "date": dates.date,
            "variable": variable,
            "value": values.round(2),
            "unit": unit,
            "consistency_level": "2",
            "source_status": "1",
            "source": "exemplo sintético",
        }
    )

daily = pd.concat(
    [
        frame("discharge", "m3/s", base=120, amp=90, noise=15),
        frame("stage", "cm", base=250, amp=120, noise=20),
        frame("rainfall", "mm", base=4, amp=4, noise=3),
    ],
    ignore_index=True,
)
daily.groupby("variable")["value"].describe()[["mean", "min", "max"]]

In [ ]:
# Disponibilidade anual
ph.analyze_hydro_data(daily, "daily_availability", variable="discharge").head()

In [ ]:
# Curva de permanência (posições de Weibull) para alguns percentis
fdc = ph.analyze_hydro_data(
    daily, "flow_duration", variable="discharge",
    exceedance_probabilities=[5, 10, 50, 90, 95],
)
fdc[["permanence_pct", "value"]]

In [ ]:
# Índices de vazão: QMLT (média de longo período), Q90 e Q95
ph.analyze_hydro_data(daily, "flow_indices", variable="discharge")

In [ ]:
# Máximas anuais com flags de qualidade e ranking (para análise de frequência)
maxima = ph.analyze_hydro_data(daily, "annual_maxima", variable="discharge")
maxima[["analysis_year", "max_value", "max_date", "n_flags", "rank_descending"]]

In [ ]:
# Índices de chuva ETCCDI (Rx1day, Rx5day, R10/R20, SDII, CDD, CWD)
rain = ph.analyze_hydro_data(daily, "rainfall_indices", variable="rainfall")
rain[["analysis_year", "total_rainfall", "rx1day", "rx5day", "r10", "sdii", "cdd", "cwd"]]

## 8. Diagnósticos fluviométricos (offline, banco interno)

`diagnose_station_hydrometry` avalia medições de descarga e curvas-chave de uma estação.
Quando você não fornece dados próprios, o banco interno (retrato 2026-06) é usado como
referência e a origem fica registrada no resultado.

In [ ]:
diag = ph.diagnose_station_hydrometry(station_code="34879500")
print("origem da referência:", diag["hydrometry_reference_source"])
print("retrato:", diag["hydrometry_reference_snapshot"])
diag["summary"]

`diagnose_daily_hydrometry` cruza vazão × cota × curva-chave dia a dia (útil com uma
série diária real e as curvas-chave da estação):

```python
daily_real = ph.get_ana_data("daily_discharge", station_code="56460000",
                             start_date="2018-01-01", end_date="2018-12-31")
diag_daily = ph.diagnose_daily_hydrometry(daily_real, station_code="56460000")
diag_daily["summary"]
```

## 9. Gráficos (offline; requer o extra `plot`)

`plot_hydro_data` retorna um objeto `matplotlib.figure.Figure`. Envolvemos em
`try/except` para o tutorial não falhar caso o matplotlib não esteja instalado.

In [ ]:
try:
    fig = ph.plot_hydro_data(daily, "flow_duration", variable="discharge")
    fig.suptitle("Curva de permanência (exemplo sintético)")
    display(fig)
except ph.PyHidroWebError as exc:
    print("Gráfico indisponível:", exc)

Outros gráficos: `daily_series` (chuva em barras), `daily_availability` (heatmap
mensal), `monthly_summary`, `annual_summary`, `rating_curves`, `measurements`,
`annual_maxima`, `low_flows`, `rainfall_indices`, `cross_section_profile`.

## 10. Tabelas e exportação (offline)

In [ ]:
# Qualquer análise também está disponível como tabela via table_hydro_data
ph.table_hydro_data(daily, "flow_indices", variable="discharge")

In [ ]:
# Exportar vários componentes de uma vez (CSV ou Parquet, com manifesto)
import tempfile, os

out_dir = os.path.join(tempfile.mkdtemp(), "export")
written = ph.write_hydro_data(
    {
        "serie_diaria": daily,
        "curva_permanencia": ph.analyze_hydro_data(daily, "flow_duration", variable="discharge"),
        "indices_vazao": ph.analyze_hydro_data(daily, "flow_indices", variable="discharge"),
    },
    out_dir,
    format="csv",
    manifest=True,
)
sorted(p.name for p in written.values())

Colunas com nomes sensíveis (token/senha/cpf/…) são removidas automaticamente antes da
escrita.

## 11. Tratamento de erros

Todas as exceções herdam de `PyHidroWebError`:

In [ ]:
from pyhydroweb.exceptions import InvalidDataTypeError

try:
    ph.analyze_hydro_data(daily, "analise_inexistente")
except ValueError as exc:
    print("análise inválida:", exc)

try:
    ph.get_ana_data("produto_inexistente")
except ValueError as exc:
    print("produto inválido:", exc)

## 12. API legada (depreciada)

As funções pré-2.0 (`download_hidroweb_data`, `extract_data`, `download_from_list`,
`download_from_shape`) ainda funcionam como wrappers depreciados sobre os novos módulos
(agora via HTTPS e com o contrato padronizado), emitindo `DeprecationWarning`. Migre
para `get_ana_data`/`get_ana_data_batch`.

## Apêndice: Logging

O pyHidroWeb usa o módulo `logging` padrão. Ative mensagens de depuração com:

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)
logging.getLogger("pyhydroweb").setLevel(logging.DEBUG)
print("logging configurado")